# 03 — Modélisation & Évaluation

> **Objectif** : Entraîner et comparer plusieurs modèles de classification IT,  
> évaluer le pipeline complet (ML + heuristiques), et viser **F1-macro ≥ 0.87**.
>
> **Pipeline** :
> 1. Chargement des données prétraitées
> 2. Split sans fuite (GroupShuffleSplit sur textes uniques)
> 3. Modèles comparés : Dummy · LR+SMOTE · LinearSVC · LightGBM · Ensemble
> 4. Évaluation avec heuristiques (pipeline complet)
> 5. Tableau de benchmark + figures PNG
> 6. Analyse de confiance (HITL)
> 7. Sauvegarde du meilleur modèle + métadonnées

## 1. Imports & Configuration

In [ ]:
import os, sys, json, warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
os.chdir(ROOT)
warnings.filterwarnings('ignore')
print(f'Racine projet : {ROOT}')

import pandas as pd
import numpy as np
import yaml
import joblib
import sklearn
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.sklearn

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
    f1_score, accuracy_score
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

try:
    import lightgbm as lgb
    HAS_LGBM = True
    print(f'LightGBM {lgb.__version__} disponible')
except ImportError:
    HAS_LGBM = False
    print('LightGBM non installé — ignoré (pip install lightgbm)')

matplotlib.rcParams['figure.dpi'] = 130
plt.style.use('seaborn-v0_8-whitegrid')

FIGURES_DIR = ROOT / 'notebooks' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# --- Correction MLflow ---
(ROOT / 'mlruns').mkdir(parents=True, exist_ok=True)          # <-- ajout
mlflow.set_tracking_uri((ROOT / 'mlruns').as_uri())           # <-- modifié

mlflow.set_experiment('helpdesk-classification')

with open(ROOT / 'params.yaml') as f:
    params = yaml.safe_load(f)

print(f'sklearn {sklearn.__version__}')
print(f'MLflow tracking : {ROOT}/mlruns')

## 2. Chargement & Split anti-fuite (GroupShuffleSplit par Titre)

Le corpus contient **58 types de tickets récurrents** — chaque Titre est lié à une unique
catégorie (vérifiéexpérimentalement : 0 Titre avec plus d'une catégorie).
Un split naïf sur `text_clean` laisserait les mêmes **Titres** en train et en test :
le modèle mémoriserait la correspondance `Titre → Catégorie` → F1 artificiellement parfait.

**Correctif appliqué** : `StratifiedShuffleSplit` sur les **Titres uniques**.
Tous les tickets d'un même type (même Titre) tombent dans le **même** split.
Le modèle doit donc généraliser à des Titres **jamais vus** en entraînement.

In [ ]:
df = pd.read_csv(ROOT / params['data']['processed_path'])
print(f'Dataset : {len(df)} tickets | {df["Catégorie"].nunique()} classes')
print(f'Titres uniques : {df["Titre"].nunique()} (chacun lié à 1 seule catégorie)')

# ── Split anti-fuite par Titre ────────────────────────────────────────────────
# Chaque Titre mappe 1:1 sur une Catégorie → on splitte les Titres, pas les textes.
# Ainsi aucun Titre de test n'est visible en entraînement.
titres_uniq = df.groupby('Titre')['Catégorie'].first().reset_index()

sss = StratifiedShuffleSplit(
    n_splits=1,
    test_size=params['data']['test_size'],
    random_state=params['data']['random_state']
)
for tr_idx, te_idx in sss.split(titres_uniq['Titre'], titres_uniq['Catégorie']):
    train_titres = set(titres_uniq.iloc[tr_idx]['Titre'])
    test_titres  = set(titres_uniq.iloc[te_idx]['Titre'])

train_df = df[df['Titre'].isin(train_titres)].copy()
test_df  = df[df['Titre'].isin(test_titres)].copy()

X_train = train_df['text_clean']
y_train = train_df['Catégorie']
X_test  = test_df['text_clean']
y_test  = test_df['Catégorie']

# Sauvegarder les splits
(ROOT / 'data' / 'splits').mkdir(parents=True, exist_ok=True)
train_df.to_csv(ROOT / params['data']['train_path'], index=False)
test_df.to_csv(ROOT  / params['data']['test_path'],  index=False)
rag_df = df[df['STATUT'] == 'Résolu'].reset_index(drop=True)
rag_df.to_csv(ROOT / params['data']['rag_path'], index=False)

titre_overlap = train_titres & test_titres
print(f'\n✅ Overlap Titres train∩test : {len(titre_overlap)} (doit être 0)')
print(f'   Train : {len(X_train)} tickets | {len(train_titres)} Titres distincts')
print(f'   Test  : {len(X_test)}  tickets | {len(test_titres)} Titres distincts')
print(f'   RAG   : {len(rag_df)} tickets résolus')
print(f'\n🔍 Titres en test (généralization cible) :')
for t in sorted(test_titres):
    cat = titres_uniq[titres_uniq['Titre'] == t]['Catégorie'].values[0]
    print(f'   [{cat}] {t}')

## 3. Entraînement multi-modèles

Modèles comparés :
| Modèle | Justification |
|---|---|
| Dummy (most_frequent) | Référence plancher |
| LR + SMOTE | Baseline actuelle (F1=0.77) |
| LinearSVC calibré | Meilleur choix usuel pour TF-IDF (marges max) |
| LightGBM | Gradient boosting — robuste au déséquilibre |
| Ensemble (vote) | Combine LR + LinearSVC pour stabilité |

In [ ]:
NLP = params['nlp']
RS  = params['data']['random_state']

def make_tfidf(**kwargs):
    return TfidfVectorizer(
        max_features = NLP['max_features'],
        ngram_range  = tuple(NLP['ngram_range']),
        min_df       = NLP['min_df'],
        sublinear_tf = True,
        **kwargs
    )

results  = {}
pipelines = {}

def evaluate(name, pipeline, X_tr, y_tr, X_te, y_te, use_proba=True):
    """Entraîne, évalue et loggue dans MLflow."""
    with mlflow.start_run(run_name=name):
        pipeline.fit(X_tr, y_tr)
        y_pred = pipeline.predict(X_te)
        f1  = f1_score(y_te, y_pred, average='macro')
        acc = accuracy_score(y_te, y_pred)
        mlflow.log_metric('f1_macro', f1)
        mlflow.log_metric('accuracy', acc)
        mlflow.log_param('model', name)
        print(f'\n{"="*55}  {name}')
        print(f'F1-macro : {f1:.4f}  |  Accuracy : {acc:.4f}')
        print(classification_report(y_te, y_pred))
        results[name]   = {'f1': f1, 'acc': acc, 'pred': y_pred}
        pipelines[name] = pipeline
    return pipeline, f1, y_pred

# ── 1. Dummy ─────────────────────────────────────────────────────────────────
dummy = DummyClassifier(strategy='most_frequent', random_state=RS)
dummy.fit(X_train, y_train)
y_dummy = dummy.predict(X_test)
f1_dummy = f1_score(y_test, y_dummy, average='macro')
results['Dummy'] = {'f1': f1_dummy, 'acc': accuracy_score(y_test, y_dummy), 'pred': y_dummy}
print(f'Dummy F1-macro : {f1_dummy:.4f}')

In [ ]:
# ── 2. Logistic Regression + SMOTE (baseline actuelle) ───────────────────────
pipe_lr = ImbPipeline([
    ('tfidf', make_tfidf()),
    ('smote', SMOTE(random_state=RS, k_neighbors=3)),
    ('clf',   LogisticRegression(C=1.0, max_iter=1000,
                                 class_weight='balanced', random_state=RS)),
])
evaluate('LR_SMOTE', pipe_lr, X_train, y_train, X_test, y_test)

In [ ]:
# ── 3. LinearSVC calibré (attendu meilleur pour TF-IDF) ──────────────────────
# LinearSVC donne de meilleures marges sur les features sparses TF-IDF.
# CalibratedClassifierCV lui ajoute predict_proba() (nécessaire pour HITL).
pipe_svc = SkPipeline([
    ('tfidf', make_tfidf()),
    ('clf',   CalibratedClassifierCV(
        LinearSVC(C=1.0, max_iter=2000, class_weight='balanced', random_state=RS),
        cv=3
    )),
])
evaluate('LinearSVC_calibré', pipe_svc, X_train, y_train, X_test, y_test)

In [ ]:
# ── 4. LightGBM (si disponible) ──────────────────────────────────────────────
if HAS_LGBM:
    from lightgbm import LGBMClassifier
    pipe_lgbm = SkPipeline([
        ('tfidf', make_tfidf()),
        ('clf',   LGBMClassifier(
            n_estimators   = 300,
            learning_rate  = 0.1,
            class_weight   = 'balanced',
            random_state   = RS,
            verbose        = -1,
            n_jobs         = -1,
        )),
    ])
    evaluate('LightGBM', pipe_lgbm, X_train, y_train, X_test, y_test)
else:
    print('LightGBM ignoré — non installé')

In [ ]:
# ── 5. Ensemble LR + LinearSVC (vote souple) ─────────────────────────────────
pipe_ensemble = SkPipeline([
    ('tfidf', make_tfidf()),
    ('vote',  VotingClassifier(
        estimators=[
            ('lr',  LogisticRegression(C=1.0, max_iter=1000,
                                       class_weight='balanced', random_state=RS)),
            ('svc', CalibratedClassifierCV(
                LinearSVC(C=1.0, max_iter=2000,
                          class_weight='balanced', random_state=RS), cv=3)),
        ],
        voting='soft'
    )),
])
evaluate('Ensemble_LR_SVC', pipe_ensemble, X_train, y_train, X_test, y_test)

## 4. Évaluation du pipeline complet : ML + Heuristiques

Le pipeline de production applique les **heuristiques** après la prédiction ML :  
- `FORCE_RULES` : « outlook » → Microsoft toujours (quel que soit le score ML)  
- `SOFT_RULES`  : correctifs si confiance < seuil (dossier sécurisé → Sécurité, etc.)

Cette cellule mesure le **F1 réel du système** (ce que voit l'utilisateur en production).

In [ ]:
from models.heuristics import apply_heuristics

THRESHOLD = 0.40

def eval_with_heuristics(pipeline, test_df, y_test, name, has_proba=True):
    """
    Évalue le pipeline complet : prédiction ML + heuristiques.
    Utilise les colonnes Titre + Description originales pour les heuristiques.
    """
    y_final = []
    X_clean = test_df['text_clean'].values
    titres  = test_df['Titre'].values if 'Titre' in test_df.columns else [''] * len(test_df)
    descs   = test_df['Description'].values if 'Description' in test_df.columns else X_clean

    for i in range(len(X_clean)):
        ml_pred = pipeline.predict([X_clean[i]])[0]
        if has_proba:
            ml_conf = float(pipeline.predict_proba([X_clean[i]])[0].max())
        else:
            ml_conf = 0.50
        original_text = f"{titres[i]} {descs[i]}"
        h = apply_heuristics(original_text, ml_pred, ml_conf, THRESHOLD)
        y_final.append(h['categorie'])

    f1_h  = f1_score(y_test, y_final, average='macro')
    acc_h = accuracy_score(y_test, y_final)
    print(f'\n{"="*55}  {name} + Heuristiques')
    print(f'F1-macro : {f1_h:.4f}  |  Accuracy : {acc_h:.4f}')
    print(classification_report(y_test, y_final))
    return f1_h, y_final

# Évaluer le meilleur modèle ML seul
best_ml_name = max((k for k in results if k in pipelines), key=lambda k: results[k]['f1'])
best_pipe    = pipelines[best_ml_name]
print(f'Meilleur modèle ML seul : {best_ml_name} — F1={results[best_ml_name]["f1"]:.4f}')

f1_with_h, y_pred_final = eval_with_heuristics(
    best_pipe, test_df, y_test, best_ml_name
)

results[f'{best_ml_name}+Heuristiques'] = {
    'f1' : f1_with_h,
    'acc': accuracy_score(y_test, y_pred_final),
    'pred': y_pred_final
}

## 5. Tableau de benchmark

In [ ]:
print('\n' + '=' * 65)
print('TABLEAU DE BENCHMARK — HelpDesk AI')
print('=' * 65)
print(f'{"Modèle":<35} {"F1-macro":>9}  {"Accuracy":>9}')
print('-' * 65)
for name, res in sorted(results.items(), key=lambda x: x[1]['f1']):
    marker = ' ← MEILLEUR' if res['f1'] == max(r['f1'] for r in results.values()) else ''
    print(f'{name:<35} {res["f1"]:>9.4f}  {res["acc"]:>9.4f}{marker}')
print('=' * 65)
print(f'Objectif jury : F1-macro ≥ 0.87')
print(f'Atteint       : {"✅" if max(r["f1"] for r in results.values()) >= 0.87 else "⚠️ pas encore"}')

In [ ]:
# Figure — Benchmark bar chart
fig, ax = plt.subplots(figsize=(11, 5))
names   = list(results.keys())
f1s     = [results[n]['f1'] for n in names]
colors  = ['#C44E52' if 'Heuristique' in n else '#4C72B0' for n in names]

bars = ax.barh(names, f1s, color=colors, edgecolor='white', height=0.6)
ax.axvline(0.87, color='orange', linestyle='--', linewidth=1.5, label='Objectif ≥ 0.87')
for bar, val in zip(bars, f1s):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=10)
ax.set_xlim(0, 1.05)
ax.set_xlabel('F1-macro', fontsize=11)
ax.set_title('Benchmark des modèles — HelpDesk AI\n'
             '(rouge = pipeline complet avec heuristiques)', fontweight='bold')
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'mod_01_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ mod_01_benchmark.png')

## 6. Matrices de confusion — sauvegardées en PNG

In [ ]:
def save_confusion_matrix(y_true, y_pred, classes, title, filename):
    cm = confusion_matrix(y_true, y_pred, labels=classes)
    cm_df = pd.DataFrame(cm, index=classes, columns=classes)
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues',
                linewidths=0.5, linecolor='white', ax=ax, cbar=True)
    ax.set_title(title, fontsize=13, fontweight='bold', pad=15)
    ax.set_ylabel('Réel', fontsize=11, fontweight='bold')
    ax.set_xlabel('Prédit', fontsize=11, fontweight='bold')
    plt.tight_layout()
    path = FIGURES_DIR / filename
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ {filename}')
    return path

classes = sorted(y_test.unique())

# Matrice — meilleur modèle ML seul
save_confusion_matrix(
    y_test,
    results[best_ml_name]['pred'],
    classes,
    f'Matrice de Confusion — {best_ml_name}\n(ML seul, F1={results[best_ml_name]["f1"]:.4f})',
    'mod_02_confusion_ml.png'
)

# Matrice — pipeline complet avec heuristiques
save_confusion_matrix(
    y_test,
    y_pred_final,
    classes,
    f'Matrice de Confusion — {best_ml_name} + Heuristiques\n(F1={f1_with_h:.4f})',
    'mod_03_confusion_heuristiques.png'
)

## 7. Analyse de confiance (HITL — Human-in-the-Loop)

Le seuil de confiance détermine quels tickets sont routés automatiquement  
vs envoyés à la revue humaine.  
**Objectif** : maximiser la couverture tout en maintenant 100 % de précision.

In [ ]:
y_proba  = best_pipe.predict_proba(X_test)
y_pred_v = best_pipe.predict(X_test)
conf_max = y_proba.max(axis=1)
correct  = (y_pred_v == y_test.values)

# Analyse par tranches
bins   = [0, 0.40, 0.60, 0.80, 1.01]
labels = ['<40%', '40–60%', '60–80%', '>80%']
df_conf = pd.DataFrame({'conf': conf_max, 'correct': correct})
df_conf['tranche'] = pd.cut(conf_max, bins=bins, labels=labels)

print('Distribution des scores de confiance et précision par tranche :')
print(f'{'Tranche':<12} {'Tickets':>8} {'% total':>8} {'Précision':>10}')
print('-' * 45)
for t in labels:
    sub = df_conf[df_conf['tranche'] == t]
    if len(sub):
        pct = len(sub) / len(df_conf) * 100
        prec = sub['correct'].mean()
        print(f'{t:<12} {len(sub):>8}  {pct:>7.1f}%  {prec:>10.2%}')

THRESHOLD = 0.40
haute = df_conf[df_conf['conf'] >= THRESHOLD]
basse = df_conf[df_conf['conf'] <  THRESHOLD]

print(f'\nAvec seuil = {THRESHOLD:.0%} :')
print(f'  Auto-routé (haute conf.) : {len(haute)} tickets ({len(haute)/len(df_conf)*100:.1f}%) '
      f'→ précision {haute["correct"].mean():.2%}')
print(f'  Revue humaine (HITL)     : {len(basse)} tickets ({len(basse)/len(df_conf)*100:.1f}%)')

In [ ]:
# Figure — Distribution des confiances
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(conf_max, bins=25, color='#4C72B0', edgecolor='white', alpha=0.85)
axes[0].axvline(THRESHOLD, color='orange', linestyle='--', linewidth=2,
                label=f'Seuil HITL = {THRESHOLD}')
axes[0].set_xlabel('Score de confiance ML')
axes[0].set_title('Distribution de la confiance', fontweight='bold')
axes[0].legend()

# Courbe couverture vs précision
thresholds = np.linspace(0, 1, 50)
coverages  = []
precisions = []
for th in thresholds:
    mask = conf_max >= th
    if mask.sum() > 0:
        coverages.append(mask.mean())
        precisions.append(correct[mask].mean())
    else:
        coverages.append(0)
        precisions.append(1)

axes[1].plot(thresholds, precisions, color='#55A868', linewidth=2, label='Précision')
axes[1].plot(thresholds, coverages,  color='#4C72B0', linewidth=2, label='Couverture')
axes[1].axvline(THRESHOLD, color='orange', linestyle='--', linewidth=2,
                label=f'Seuil = {THRESHOLD}')
axes[1].set_xlabel('Seuil de confiance')
axes[1].set_title('Précision vs Couverture (HITL)', fontweight='bold')
axes[1].legend()

fig.suptitle('Analyse HITL — Human-in-the-Loop', fontsize=12, fontweight='bold')
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'mod_04_hitl_analyse.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ mod_04_hitl_analyse.png')

## 8. Analyse des erreurs du meilleur modèle

In [ ]:
test_df_err = test_df.copy()
test_df_err['pred_ml']   = results[best_ml_name]['pred']
test_df_err['pred_final']= y_pred_final
test_df_err['correct_ml']    = test_df_err['Catégorie'] == test_df_err['pred_ml']
test_df_err['correct_final'] = test_df_err['Catégorie'] == test_df_err['pred_final']

# Tickets corrigés par les heuristiques
corriges = test_df_err[
    ~test_df_err['correct_ml'] & test_df_err['correct_final']
]
print(f'Tickets corrigés par les heuristiques : {len(corriges)}')
if len(corriges) > 0:
    print(corriges[['Titre', 'Catégorie', 'pred_ml', 'pred_final']]
          .drop_duplicates(subset=['Titre']).to_string())

# Erreurs résiduelles après heuristiques
erreurs = test_df_err[~test_df_err['correct_final']]
print(f'\nErreurs résiduelles : {len(erreurs)} tickets')
if len(erreurs) > 0:
    print(erreurs[['Titre', 'Catégorie', 'pred_final']]
          .drop_duplicates(subset=['Titre']).to_string())

## 9. Sauvegarde du meilleur modèle + métadonnées

In [ ]:
# Sélectionner le meilleur modèle (pipeline complet avec heuristiques)
best_final_name = max(results, key=lambda k: results[k]['f1'])
best_pipe_key   = best_final_name.replace('+Heuristiques', '').strip()
best_pipe_save  = pipelines.get(best_pipe_key, best_pipe)

# Sauvegarder le pipeline
model_save_path = ROOT / 'models' / 'best_pipeline.pkl'
joblib.dump(best_pipe_save, model_save_path)
print(f'✅ Pipeline sauvegardé : {model_save_path}')
print(f'   Modèle              : {best_pipe_key}')
print(f'   sklearn version     : {sklearn.__version__}')

# Test de rechargement
p_loaded = joblib.load(model_save_path)
test_text = ['outlook affiche erreur activation licence microsoft']
print(f'\n🧪 Test rechargement :')
print(f'   Texte   : {test_text[0]}')
print(f'   Prédit  : {p_loaded.predict(test_text)[0]}')
print(f'   Conf.   : {p_loaded.predict_proba(test_text).max():.2%}')

In [ ]:
# Métadonnées du modèle
f1_best_ml   = results[best_pipe_key]['f1']
f1_best_full = results[best_final_name]['f1']

meta = {
    'model_name'              : best_pipe_key,
    'sklearn_version'         : sklearn.__version__,
    'f1_macro_ml_seul'        : round(f1_best_ml, 4),
    'f1_macro_avec_heuristiques': round(f1_best_full, 4),
    'f1_macro_cv'             : None,
    'accuracy_test'           : round(results[best_pipe_key]['acc'], 4),
    'confidence_threshold'    : THRESHOLD,
    'classes'                 : sorted(y_test.unique().tolist()),
    'train_size'              : len(X_train),
    'test_size'               : len(X_test),
    'unique_train'            : int(X_train.nunique()),
    'unique_test'             : int(X_test.nunique()),
    'dataset_note'            : 'Corpus organisé en ~58 types de tickets récurrents ; '
                                'split par groupes de textes uniques (0 fuite de données)',
    'known_weaknesses'        : [
        'Sécurité : recall faible sur ML seul — compensé par heuristiques (dossier sécurisé)',
        'Microsoft : confusion Applicatifs sur tickets Outlook — corrigé par FORCE_RULES',
    ],
    'confidence_stats': {
        'haute_confiance_pct'      : round(len(haute) / len(df_conf) * 100, 1),
        'haute_confiance_precision': round(float(haute['correct'].mean()), 4),
        'basse_confiance_pct'      : round(len(basse) / len(df_conf) * 100, 1),
        'interpretation'           : f'{len(haute)/len(df_conf)*100:.1f}% routage automatique '
                                     f'à {haute["correct"].mean():.0%} de précision. '
                                     f'{len(basse)/len(df_conf)*100:.1f}% envoyés en revue humaine.',
    },
}

meta_path = ROOT / 'models' / 'model_metadata.json'
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print('✅ Métadonnées sauvegardées : models/model_metadata.json')
print(json.dumps(meta, ensure_ascii=False, indent=2))

In [ ]:
# Résumé final
print('=' * 65)
print('RÉSUMÉ — MODÉLISATION HELPDESK AI')
print('=' * 65)
print(f'  Meilleur modèle ML      : {best_pipe_key}')
print(f'  F1-macro ML seul        : {f1_best_ml:.4f}')
print(f'  F1-macro + heuristiques : {f1_best_full:.4f}  {'✅' if f1_best_full >= 0.87 else '⚠️'}')
print(f'  Objectif jury (≥ 0.87)  : {'ATTEINT ✅' if f1_best_full >= 0.87 else 'Non atteint ⚠️'}')
print(f'  Routage auto (HITL)     : {len(haute)/len(df_conf)*100:.1f}% @ '
      f'{haute["correct"].mean():.0%} précision')
print()
print('  Figures sauvegardées dans notebooks/figures/ :')
for f in sorted(FIGURES_DIR.glob('mod_*.png')):
    print(f'    - {f.name}')
print()
print('  → Étape suivante : 04_rag.ipynb (évaluation RAG + RAGAS)')

## 10. Interpretation — resultats honnetes pour le jury

### Ce que montrent ces chiffres

| Evaluation | F1-macro | Interpretation |
|---|---|---|
| **ML seul** (sans heuristiques) | **0.38** | Generalisation a 12 Titres jamais vus en entrainement |
| **ML + Heuristiques** | **0.57** | FORCE_RULES sauve Microsoft (0.00 -> 0.86 F1) |
| Benchmark externe (04) | **0.83** | Meme pipeline, donnees publiques reelles (type ITIL) |

### Pourquoi F1=0.38 est honnete — et pas un echec

Le split **GroupShuffleSplit par Titre** (46 Titres train / 12 Titres test) garantit
qu'aucun Titre du test n'a ete vu a l'entrainement. Le modele TF-IDF doit donc
generaliser a des **types de tickets entierement nouveaux** (tache zero-shot par Titre).

### Ce que ca revele

1. **Les heuristiques ont de la valeur** : Microsoft passe de 0.00 a 0.86 grace aux FORCE_RULES.

2. **Communication vs Microsoft** : "Probleme Microsoft Teams" est categorie [Communication]
   mais les FORCE_RULES le redirigent vers Microsoft — conflit semantique reel.

3. **Motivation CamemBERT (Phase 5)** : un modele de langage comprend le sens sans
   dependre des Titres exacts -> meilleure generalisation attendue.

### Synthese pour la soutenance

> *"Notre pipeline TF-IDF atteint F1=0.57 sur des types de tickets jamais vus
> (evaluation hors-distribution stricte, GroupShuffleSplit par Titre).
> Les heuristiques metier compensent les lacunes du ML sur les categories IT structurees.
> Le benchmark externe (notebook 04) valide la methodologie a F1=0.83 sur un corpus
> reel public — le modele generalise bien avec suffisamment de types de tickets.*"